# Demo: Fitting ARIMA and SARIMAX with statsmodels

A regional electric utility needs to forecast monthly electricity demand. They have eight years of data plus average temperatures. We're going to fit two models -- a plain ARIMA and a SARIMAX with temperature -- and see how much that extra information helps.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
df = pd.read_csv("../data/electricity_demand.csv", parse_dates=["date"], index_col="date")
df = df.asfreq("MS")

# Train on everything through 2023, test on 2024
train = df.loc[:"2023-12-01"]
print(f"Data: {len(df)} months")
print(f"Train: {len(train)} months, Test: {len(df) - len(train)} months")

## ACF/PACF

Before picking model orders, look at the autocorrelation structure. We seasonally difference first (lag 12) to remove the annual cycle, then check what's left.

In [ ]:
diffed = train["demand_mwh"].diff(12).dropna()
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(diffed, ax=axes[0], lags=36)
plot_pacf(diffed, ax=axes[1], lags=36)
axes[0].set_title("ACF (seasonally differenced)")
axes[1].set_title("PACF (seasonally differenced)")
plt.tight_layout()
plt.show()

The ACF spike at lag 1 suggests an MA(1) component. The seasonal spikes at lag 12 confirm we need a seasonal specification. This points us toward something like SARIMAX(1,1,1)(1,1,1,12).

## ARIMA(2,1,1) -- No Seasonality

Let's start simple. A plain ARIMA with no seasonal component and no exogenous variables.

In [ ]:
arima = SARIMAX(train["demand_mwh"], order=(2, 1, 1)).fit(disp=False)
arima_fc = arima.get_forecast(12)
arima_mean = arima_fc.predicted_mean
arima_ci = arima_fc.conf_int()
print(f"AIC: {arima.aic:.1f}")

## SARIMAX(1,1,1)(1,1,1,12) + Temperature

Now let's add the seasonal component and temperature as an exogenous variable. Demand is U-shaped with temperature -- high in summer (AC) and winter (heating). That's useful information.

In [ ]:
sarimax = SARIMAX(
    train["demand_mwh"],
    exog=train["avg_temp_f"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12)
).fit(disp=False)

# For the forecast, we need future temperatures. Use seasonal averages from training data.
seasonal_temp = train.groupby(train.index.month)["avg_temp_f"].mean()
future_temp = pd.Series(
    [seasonal_temp[m] for m in range(1, 13)],
    index=pd.date_range("2024-01-01", periods=12, freq="MS")
)

sarimax_fc = sarimax.get_forecast(12, exog=future_temp.values)
sarimax_mean = sarimax_fc.predicted_mean
sarimax_ci = sarimax_fc.conf_int()
print(f"AIC: {sarimax.aic:.1f}")

AIC dropped substantially. Lower is better. Temperature is actually helping the model explain the variance.

## Comparing the Forecasts

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

for ax, name, mean, ci, color in [
    (axes[0], "ARIMA(2,1,1)", arima_mean, arima_ci, "tab:blue"),
    (axes[1], "SARIMAX + temperature", sarimax_mean, sarimax_ci, "tab:orange"),
]:
    ax.plot(train.index, train["demand_mwh"].values, color="black", label="Historical")
    ax.plot(mean.index, mean.values, color=color, linestyle="--", linewidth=2, label="Forecast")
    ax.fill_between(mean.index, ci.iloc[:, 0], ci.iloc[:, 1], color=color, alpha=0.15, label="95% interval")
    ax.set_ylabel("Demand (MWh)")
    ax.set_title(name)
    ax.legend(loc="upper left")

plt.tight_layout()
plt.show()

Look at the prediction intervals. ARIMA's are wide -- it's uncertain because it can't model the annual cycle. SARIMAX's are tighter because the seasonal component and temperature explain a lot of the variation. For the grid operators who need to plan capacity, those tighter intervals are the difference between a useful forecast and a useless one.